# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. `PinchWorkspace` keeps named study cases, while each case remains a real `PinchProblem` with the familiar `target`, `plot`, `summary_frame`, and `compare_to` workflow.


In [46]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchWorkspace

In [47]:
workspace = PinchWorkspace(
    source="crude_preheat_train.json",
    project_name="crude_preheat_train",
)
baseline = workspace.use_case("baseline")

summary = baseline.summary_frame()
catalog = baseline.plot.catalog()

{
    "cases": workspace.list_cases(),
    "active_case": workspace.active_case_name,
    "graph_count": len(catalog),
    "zone_tree_present": "zone_tree" in workspace.get_case_payload("baseline"),
}

{'cases': ['baseline'],
 'active_case': 'baseline',
 'graph_count': 14,
 'zone_tree_present': True}

In [48]:
summary

,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,crude_preheat_train/Direct Integration,749.999995,1000.0,5150.000005,145.0,145.0,MP Steam: 750.00,Cooling Water: 1000.00
1,crude_preheat_train/Total Process Target,749.999995,1000.0,5150.000005,NaN,NaN,MP Steam: 750.00,Cooling Water: 1000.00
2,crude_preheat_train/Total Site Target,749.999995,1000.0,5150.000005,258.9,11.1,MP Steam: 750.00,Cooling Water: 1000.00
3,Crude Unit/Direct Integration,749.999995,1000.0,5150.000005,145.0,145.0,MP Steam: 750.00,Cooling Water: 1000.00


## Baseline graphs from the built-in plot accessor
Because each workspace case is a real `PinchProblem`, the notebook can use the normal `plot` accessor directly instead of rebuilding figures from serialized helper payloads.


In [49]:
catalog.loc[
    catalog["Target"] == "crude_preheat_train/Direct Integration",
    ["Target", "Graph Type", "Graph Name"],
].reset_index(drop=True)

,Target,Graph Type,Graph Name
0,crude_preheat_train/Direct Integration,Composite Curves,Graph 1
1,crude_preheat_train/Direct Integration,Shifted Composite Curves,Graph 2
2,crude_preheat_train/Direct Integration,Balanced Composite Curves,Graph 3
3,crude_preheat_train/Direct Integration,Grand Composite Curve,Graph 4
4,crude_preheat_train/Direct Integration,Net Load Profiles,Graph 5
5,crude_preheat_train/Direct Integration,Grand Composite Curve with Heat Pump,Graph 6


In [50]:
baseline.plot.composite_curve(zone_name="crude_preheat_train")

In [51]:
baseline.plot.shifted_composite_curve(
    zone_name="crude_preheat_train",
)

In [52]:
baseline.plot.grand_composite_curve(
    zone_name="crude_preheat_train/Direct Integration",
)

## One named case before the sweep
Clone the baseline into a named case, adjust the root `dt_cont_multiplier`, and compare the two `PinchProblem` summaries directly.


In [ ]:
wide_dt = workspace.copy_case("baseline", "wide_dt", activate=False)
wide_dt.set_dt_cont_multiplier(2.0)
workspace.compare_cases(
    "baseline",
    "wide_dt",
    target_name="crude_preheat_train/Direct Integration",
)

,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
baseline,crude_preheat_train/Direct Integration,749.999995,1000.0,5150.000005,145.0,145.0
wide_dt,crude_preheat_train/Direct Integration,1149.999995,1400.0,4750.000005,150.0,150.0
Change,crude_preheat_train/Direct Integration,400.000000,400.0,-400.000000,5.0,5.0


In [ ]:
rows = []
for factor in np.linspace(0.5, 10.0, 96):
    case_name = f"dt_mult_{factor:05.2f}".replace(".", "_")
    case = workspace.copy_case("baseline", case_name, activate=False)
    case.set_dt_cont_multiplier(float(factor))
    row = (
        case.summary_frame()
        .loc[lambda frame: frame["Target"] == "crude_preheat_train/Direct Integration"]
        .iloc[0]
    )
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

,dt_cont multiplier,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,0.5,549.999995,800.0,5350.000005,142.5,142.5
1,0.6,589.999995,840.0,5310.000005,143.0,143.0
2,0.7,629.999995,880.0,5270.000005,143.5,143.5
3,0.8,669.999995,920.0,5230.000005,144.0,144.0
4,0.9,709.999995,960.0,5190.000005,144.5,144.5
...,...,...,...,...,...,...
91,9.6,3469.999995,3720.0,2430.000005,152.0,152.0
92,9.7,3489.999995,3740.0,2410.000005,151.5,151.5
93,9.8,3509.999995,3760.0,2390.000005,151.0,151.0
94,9.9,3529.999995,3780.0,2370.000005,150.5,150.5


In [55]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
sensitivity_fig